In [16]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

Overriding of current TracerProvider is not allowed


In [ ]:
from rag_helper import RAGBase
from starter import index, client

class RAGTraced(RAGBase):

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search"):
            return super().search(query, num_results)
        
    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            ## For Q2 uncomment this chunk of code
            # usage = response.usage
            # span.set_attribute("input_tokens", usage.input_tokens)
            # span.set_attribute("output_tokens", usage.output_tokens)
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

## Q1. First trace

Count the spans in the console output - each one is a separate `ReadableSpan` entry. How many spans does the trace produce?

In [15]:
rag = RAGTraced(
    index=index,
    llm_client=client,
)
query = "How does the agentic loop keep calling the model until it stops?"

answer = rag.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xa30f8517f92e778bf1e38fdf99e6ae20",
        "span_id": "0x044d5e9d20e236a9",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xff4b7f0a404a0eef",
    "start_time": "2026-07-14T22:23:26.409396Z",
    "end_time": "2026-07-14T22:23:26.412023Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "08f9f033-2d79-440b-b68b-b798c71b4142",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xa30f8517f92e778bf1e38fdf99e6ae20",
        "span_id": "0x44aa7f6f4bf68d0a",
        "trace_state": "[]"
    },
    "kind": "SpanKind


[ ] 1

[X] 3

[ ] 5

[ ] 7


## Q2. Capturing metrics as span attributes

Now re-run the query. How many input tokens do we see?

In [7]:
rag = RAGTraced(
    index=index,
    llm_client=client,
)
query = "How does the agentic loop keep calling the model until it stops?"

answer = rag.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xd606189e3e0272e57e4a19ec012cbd84",
        "span_id": "0xe8c2df8a68c192f4",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x9c718e1ca05cdc85",
    "start_time": "2026-07-14T21:50:39.931252Z",
    "end_time": "2026-07-14T21:50:39.938072Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.43.0",
            "service.instance.id": "08f9f033-2d79-440b-b68b-b798c71b4142",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xd606189e3e0272e57e4a19ec012cbd84",
        "span_id": "0x495c8e43968fd289",
        "trace_state": "[]"
    },
    "kind": "SpanKind

Ans = 7111

[ ] 700

[X] 7000

[ ] 70000

[ ] 700000

## Q3. Span timing

I ran the Q1 code several times and on average it falls between 1000 ms to 2000 ms, so:

[ ] Under 100ms

[ ] 100-500ms

[X] 500-2000ms

[ ] Over 2000ms

## Q4. Saving traces to SQLite

Since we only can make one `TracerProvider` in each python process, we put the code for the sqllite in another file named: `q4.py`

Re-run the query from Q1. Which span names appear in the `spans` table?

[ ] Only `rag`

[ ] `rag` and `llm`

[X] `rag`, `search`, and `llm`

[ ] `search`, `llm`, and `judge`

## Q5. Querying trace data

Using SQL (or pandas), compute the total duration for each span name
excluding `rag`. Which span type takes the most total time?

In [19]:
import sqlite3
import pandas as pd

with sqlite3.connect("traces.db") as conn:
    df = pd.read_sql_query("SELECT * FROM spans", conn)

df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784068725441174788,1784068725445502287,None,None,None
1,llm,1784068725461630277,1784068729193730665,None,None,None
2,rag,1784068725441116820,1784068729210718820,None,None,None
3,search,1784069320811037246,1784069320813138564,None,None,None
4,llm,1784069320817482353,1784069323797376140,None,None,None
5,rag,1784069320810979478,1784069323800618986,None,None,None
6,search,1784069351857026157,1784069351861559931,None,None,None
7,llm,1784069351865341502,1784069354794340414,None,None,None
8,rag,1784069351856968911,1784069354801283758,None,None,None
9,search,1784069401661653621,1784069401665544346,None,None,None


In [22]:
df["duration_ms"] = (
    df["end_time"] - df["start_time"]
) / 1e6

result = (
    df[df["name"] != "rag"]
    .groupby("name", as_index=False)
    .agg(
        span_count=("name", "count"),
        total_duration_ms=("duration_ms", "sum"),
    )
    .sort_values("total_duration_ms", ascending=False)
)

result

,name,span_count,total_duration_ms
0,llm,4,12999.184120
1,search,4,14.853316


[ ] `search`

[X] `llm`

[ ] They're all about the same

## Q6. Token stability across runs

In [27]:
import sqlite3
import pandas as pd

with sqlite3.connect("traces.db") as conn:
    df = pd.read_sql_query("SELECT * FROM spans", conn)

result = (df[
    (df['name'] == 'llm')
    & (df['input_tokens'].notnull())
    ]
    .sort_values(by='input_tokens', ascending=False)
    .head()
)
result

,name,start_time,end_time,input_tokens,output_tokens,cost
1,llm,1784070108905421950,1784070112794577146,7111.0,96.0,None
4,llm,1784070259505726788,1784070262411834794,7111.0,112.0,None
7,llm,1784070284951507849,1784070287003627921,7111.0,119.0,None
10,llm,1784070346925638074,1784070350343227370,7111.0,121.0,None


How much do the input tokens vary across these 4 runs?

[x] They're identical

[ ] Within 10% of each other

[ ] Within 50% of each other

[ ] They vary more than 50%